# 02 · torus: experimento reproducible

Paquete experimental: encoder espectral congelado → dos cabezas afines → VAE.
Las corridas se distribuyen entre dos T4, una semilla por proceso. El eigensolve de cada corrida usa una GPU; no se suman las memorias de las dos tarjetas.

**Estado:** notebook preparado, sin resultados científicos precargados. La ejecución CUDA/SM75 debe validarse en Kaggle. Internet solo es necesario si falta una dependencia o para descargar MNIST.

Adjunta el ZIP o su contenido como un Kaggle Dataset y selecciona **GPU T4 x2**. El backend `flash` compila el kernel upstream incluido para SM75; `torch` es una alternativa explícita y no se debe reportar como FlashDiffusion. FlashDiffusionTests devolvió 404 durante la preparación: se fijó la fuente pública FlashDiffusion al commit indicado en README.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## Datos y configuración

Toro plano T² en R⁴, con radios 1 y 0.6. VAE euclídeo k=2 como control de limitación topológica; no se afirma una carta global. Para el control k=4 cambia latent=4 y utiliza otro directorio de salida.

In [ ]:
cfg = json.loads((ROOT / 'config_torus.json').read_text())
# PILOTO: verifica que el pipeline funciona. No es una réplica cuantitativa del artículo.
PAPER_BUDGET = False
if PAPER_BUDGET:
    cfg.update(n=12000, modes=[2,4,8,16,32,64,128,256], epochs=200)
# Presupuesto computacional, alpha y beta forman parte del protocolo: no ajustarlos con test.
# beta=None usa la heurística de mediana de entrenamiento * beta_factor, NO beta-flow.
cfg['backend'] = 'flash'   # cambiar explícitamente a 'torch' si se desea el backend alternativo
cfg['seeds'] = [0, 1, 2]
cfg['out'] = '/kaggle/working/results_torus'
CONFIG = ROOT / 'active_config.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

## Ejecutar en dos GPUs

Cada worker verifica el kernel contra una matriz densa pequeña y controla residuos, separación de Perron y Nyström. Un fallo detiene esa corrida y queda registrado. Los logs se escriben mientras avanza el entrenamiento. No hay fallback silencioso ni plazo fijo de 120 segundos.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'launch.py'), str(CONFIG)], check=True)

## Resultados

MSE de reconstrucción fuera de muestra. El oracle predice otra variable y no se mezcla en la gráfica de MSE ambiental. Barras: desviación entre semillas, no intervalos de confianza. Inspecciona validation_mse para seleccionar configuración; reserva test para evaluación final.

In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = Path(cfg['out'])
display(pd.read_csv(RESULTS / 'metrics.csv'))
for figure in sorted(RESULTS.glob('*.png')):
    display(Image(filename=str(figure)))

## Archivos guardados

`encoder.npz`, pesos `.pt`, configuración, particiones, métricas JSON/CSV, predicciones NPZ, curvas de entrenamiento y figuras PNG/PDF. Guarda la salida de la sesión Kaggle. Cambia `out` al modificar una configuración; las corridas completas no se sobrescriben.